# 05 — Layer 2 Primary Validation: MIMIC PERform AF

The Apple Watch N=1 case study produced a null result. The research question requires a multi-subject wearable dataset with confirmed clinical labels. The MIMIC PERform AF dataset (Zenodo, 2022) satisfies all three requirements.

---

### Dataset
- **35 critically ill adults** — 19 AF, 16 NSR (pre-assigned at subject level)
- **Signal:** Finger PPG — optical sensing, the same modality as Apple Watch and all consumer wearables
- **Recording:** 20 minutes per subject at 125 Hz (~150,000 samples)
- **Ground truth:** Binary labels (AF = Abnormal, NSR = Normal) pre-assigned, no derivation required
- **Source:** Zenodo — open access, no approval required (doi:10.5281/zenodo.6807402)

### Why PPG Is the Right Signal
PPG (photoplethysmography) measures volumetric blood flow optically. It is the sensing modality in every major consumer wearable. MIMIC PERform AF bridges the gap between the clinical ECG used in training and the optical signals from devices like Apple Watch — not perfectly, but as close as any publicly available labelled dataset allows.

### Pipeline
The same 8 locked features are extracted — this time from PPG-derived RR intervals via NeuroKit2 peak detection. The same scaler (transform-only, no refit). The same SVM at the same fixed threshold (0.34). No retraining. No threshold adjustment in the primary evaluation. The question is whether the model transfers as-is.

## Cell 1 — Environment and Imports

Loads all required libraries and Layer 1 artifacts (scaler, SVM model) as read-only.
Defines directory paths for MIMIC PERform AF CSV files.

**src/ calls:** `src.mimic_perform_af_features.build_mimic_feature_matrix`
**Layer 1 artifacts:** `outputs/models/scaler.joblib`, `outputs/models/selected_model.joblib` (read-only)
**Input directories:** `data/mimic_perform_af/mimic_perform_af_csv/`, `data/mimic_perform_af/mimic_perform_non_af_csv/`

In [9]:
# src calls: src.mimic_perform_af_features.build_mimic_feature_matrix
# Layer 1 artifacts: outputs/models/scaler.joblib, outputs/models/selected_model.joblib (read-only)

import os
import numpy as np
import pandas as pd
import joblib
from scipy import stats
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, f1_score,
    recall_score, precision_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import neurokit2 as nk

import sys
PROJECT_ROOT = r'C:\Projects\GA Capstone Project'
if os.getcwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)
if os.path.join(PROJECT_ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
from mimic_perform_af_features import build_mimic_feature_matrix

# Confirm working directory
print("Working directory:", os.getcwd())

# Load Layer 1 artifacts (read-only — never overwrite)
scaler = joblib.load(os.path.join('outputs', 'models', 'scaler.joblib'))
model = joblib.load(os.path.join('outputs', 'models', 'selected_model.joblib'))

print("Scaler:", type(scaler).__name__)
print("Model:", type(model).__name__)

# Define paths
AF_DIR = os.path.join('data', 'mimic_perform_af', 'mimic_perform_af_csv')
NON_AF_DIR = os.path.join('data', 'mimic_perform_af', 'mimic_perform_non_af_csv')
LAYER2_OUT = os.path.join('outputs', 'layer2')

print(f"\nAF directory: {AF_DIR} — exists: {os.path.isdir(AF_DIR)}")
print(f"Non-AF directory: {NON_AF_DIR} — exists: {os.path.isdir(NON_AF_DIR)}")
print(f"Output directory: {LAYER2_OUT} — exists: {os.path.isdir(LAYER2_OUT)}")

Working directory: C:\Projects\GA Capstone Project
Scaler: StandardScaler
Model: SVC

AF directory: data\mimic_perform_af\mimic_perform_af_csv — exists: True
Non-AF directory: data\mimic_perform_af\mimic_perform_non_af_csv — exists: True
Output directory: outputs\layer2 — exists: True


## Cell 2 — Feature Matrix Construction

Runs the full MIMIC PERform AF pipeline: loads 35 CSV files, cleans PPG nulls via linear interpolation,
detects peaks with NeuroKit2 at 125 Hz, extracts RR intervals, computes the 8 locked features per subject.

Subjects with red quality tier (< 100 peaks) are skipped. Amber subjects (100–299 peaks) are flagged.

**src/ calls:** `src.mimic_perform_af_features.build_mimic_feature_matrix`
**Output:** `outputs/layer2/mimic_perform_af_features.csv`

In [10]:
# src calls: src.mimic_perform_af_features.build_mimic_feature_matrix
# Output: outputs/layer2/mimic_perform_af_features.csv

features_df, failed_subjects = build_mimic_feature_matrix(AF_DIR, NON_AF_DIR)

# Assertions
feature_cols = ['rmssd', 'sdnn', 'mean_rr', 'pnn50',
                'hr_mean', 'hr_std', 'rr_skewness', 'rr_kurtosis']
assert features_df.shape[1] == 11, (
    f"Expected 11 columns, got {features_df.shape[1]}"
)
assert features_df[feature_cols].isnull().sum().sum() == 0, (
    "Null values found in feature columns"
)

# Save
features_df.to_csv(
    os.path.join(LAYER2_OUT, 'mimic_perform_af_features.csv'),
    index=False
)

# Summary
print(f"\nFeature matrix saved: {len(features_df)} subjects")
print(f"Failed subjects ({len(failed_subjects)}): {failed_subjects}")
print(f"\nClass distribution:")
print(features_df['label'].value_counts().sort_index().to_string())


MIMIC PERform AF — FEATURE EXTRACTION COMPLETE
Total subjects attempted:  35
  Green (>= 300 peaks):    35
  Amber (100-299 peaks):   0
  Red / skipped:           0
Final feature matrix:      35 subjects

Class distribution:
  AF (Abnormal=1):  19
  NSR (Normal=0):   16

Feature matrix saved: 35 subjects
Failed subjects (0): []

Class distribution:
label
0    16
1    19


## Cell 3 — Gap Quantification (KS Test)

Compares the distribution of each of the 8 features between Physionet Layer 1 training data
and MIMIC PERform AF Layer 2 data using the two-sample Kolmogorov-Smirnov test.

Classifies each feature's distributional distance: Large (KS >= 0.3), Moderate (0.1–0.3), Small (< 0.1).

**Input:** `data/processed/physionet_features.csv`, `features_df` from Cell 2
**Output:** `outputs/layer2/gap_quantification_mimic.csv`

In [11]:
# src calls: none
# Input: data/processed/physionet_features.csv
# Output: outputs/layer2/gap_quantification_mimic.csv

physionet_df = pd.read_csv(
    os.path.join('data', 'processed', 'physionet_features.csv')
)

feature_cols = ['rmssd', 'sdnn', 'mean_rr', 'pnn50',
                'hr_mean', 'hr_std', 'rr_skewness', 'rr_kurtosis']

gap_results = []

for feat in feature_cols:
    physionet_vals = physionet_df[feat].dropna()
    mimic_vals = features_df[feat].dropna()

    ks_stat, p_val = stats.ks_2samp(physionet_vals, mimic_vals)

    # Determine direction
    if mimic_vals.median() > physionet_vals.median():
        direction = 'MIMIC higher'
    else:
        direction = 'MIMIC lower'

    # Classify magnitude
    if ks_stat >= 0.3:
        magnitude = 'Large'
    elif ks_stat >= 0.1:
        magnitude = 'Moderate'
    else:
        magnitude = 'Small'

    gap_results.append({
        'feature': feat,
        'ks_statistic': round(ks_stat, 4),
        'p_value': p_val,
        'direction': direction,
        'magnitude': magnitude
    })

gap_df = pd.DataFrame(gap_results)
gap_df.to_csv(
    os.path.join(LAYER2_OUT, 'gap_quantification_mimic.csv'),
    index=False
)

# Summary
large_count = (gap_df['magnitude'] == 'Large').sum()
moderate_count = (gap_df['magnitude'] == 'Moderate').sum()
small_count = (gap_df['magnitude'] == 'Small').sum()

print("Gap Quantification — KS Test Results")
print("=" * 55)
print(gap_df.to_string(index=False))
print(f"\nSummary: {large_count} large, {moderate_count} moderate, {small_count} small")

Gap Quantification — KS Test Results
    feature  ks_statistic      p_value    direction magnitude
      rmssd        0.3747 6.686197e-05 MIMIC higher     Large
       sdnn        0.3236 9.515756e-04 MIMIC higher     Large
    mean_rr        0.3853 3.646504e-05  MIMIC lower     Large
      pnn50        0.4739 1.043043e-07 MIMIC higher     Large
    hr_mean        0.3853 3.646504e-05 MIMIC higher     Large
     hr_std        0.3373 4.865331e-04 MIMIC higher     Large
rr_skewness        0.5186 3.066871e-09 MIMIC higher     Large
rr_kurtosis        0.4121 7.188761e-06 MIMIC higher     Large

Summary: 8 large, 0 moderate, 0 small


## Cell 4 — Model Application

Applies the Layer 1 SVM model to MIMIC PERform AF features. Uses `scaler.transform()` only (never
`fit_transform()`) and applies the fixed threshold of 0.34 to generate binary predictions.

No retraining, no threshold adjustment — strict Layer 1 to Layer 2 forward pass.

**Input:** `features_df` from Cell 2, `scaler` and `model` from Cell 1
**Output:** `outputs/layer2/probability_scores_mimic.csv`

In [12]:
# src calls: none
# Output: outputs/layer2/probability_scores_mimic.csv

feature_cols = ['rmssd', 'sdnn', 'mean_rr', 'pnn50',
                'hr_mean', 'hr_std', 'rr_skewness', 'rr_kurtosis']

# Scale features — transform() only, never fit_transform()
X_mimic = features_df[feature_cols].values
X_mimic_scaled = scaler.transform(X_mimic)

# Get probability scores — Abnormal class (index 1)
prob_abnormal = model.predict_proba(X_mimic_scaled)[:, 1]

# Apply fixed threshold 0.34
THRESHOLD = 0.34
predicted_label = (prob_abnormal >= THRESHOLD).astype(int)

# Add to features_df
features_df['prob_abnormal'] = np.round(prob_abnormal, 6)
features_df['predicted_label'] = predicted_label

# Save
features_df.to_csv(
    os.path.join(LAYER2_OUT, 'probability_scores_mimic.csv'),
    index=False
)

# Summary
print("Model Application — SVM at threshold 0.34")
print("=" * 55)
print(f"Mean probability:    {prob_abnormal.mean():.4f}")
print(f"Median probability:  {np.median(prob_abnormal):.4f}")
print(f"Predicted Abnormal:  {(predicted_label == 1).sum()}")
print(f"Predicted Normal:    {(predicted_label == 0).sum()}")

Model Application — SVM at threshold 0.34
Mean probability:    0.8027
Median probability:  0.8414
Predicted Abnormal:  33
Predicted Normal:    2


## Cell 5 — Evaluation Against Ground Truth

Compares SVM predictions against pre-assigned AF/NSR labels.
Computes sensitivity, specificity, AUROC, F1, and confusion matrix.

Checks pre-registered criteria: sensitivity >= 80%, specificity >= 75%.
Reports amber-tier subjects separately if any exist.

In [13]:
# src calls: none

y_true = features_df['label'].values
y_pred = features_df['predicted_label'].values
y_prob = features_df['prob_abnormal'].values

# Confusion matrix: labels [0, 1] → TN, FP, FN, TP
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
auroc = roc_auc_score(y_true, y_prob)
f1 = f1_score(y_true, y_pred)

# Pre-registered criteria
sens_pass = "PASS" if sensitivity >= 0.80 else "FAIL"
spec_pass = "PASS" if specificity >= 0.75 else "FAIL"

print("Evaluation Against Ground Truth — SVM at threshold 0.34")
print("=" * 55)
print(f"Sensitivity:   {sensitivity:.4f}  ({sensitivity*100:.1f}%)")
print(f"Specificity:   {specificity:.4f}  ({specificity*100:.1f}%)")
print(f"AUROC:         {auroc:.4f}")
print(f"F1 (Abnormal): {f1:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TP={tp}  FP={fp}")
print(f"  FN={fn}  TN={tn}")
print(f"\nPre-registered criteria:")
print(f"  Sensitivity >= 80%: {sens_pass}")
print(f"  Specificity >= 75%: {spec_pass}")

# Amber subjects — separate reporting
amber_mask = features_df['quality_tier'] == 'amber'
if amber_mask.sum() > 0:
    amber_df = features_df[amber_mask]
    amber_y_true = amber_df['label'].values
    amber_y_pred = amber_df['predicted_label'].values
    amber_correct = (amber_y_true == amber_y_pred).sum()
    print(f"\nAmber-tier subjects ({amber_mask.sum()}):")
    for _, row in amber_df.iterrows():
        match = "correct" if row['label'] == row['predicted_label'] else "incorrect"
        print(f"  {row['subject_id']}: true={int(row['label'])}, "
              f"pred={int(row['predicted_label'])}, "
              f"prob={row['prob_abnormal']:.4f} — {match}")
    print(f"  Amber accuracy: {amber_correct}/{amber_mask.sum()}")
else:
    print("\nNo amber-tier subjects.")

Evaluation Against Ground Truth — SVM at threshold 0.34
Sensitivity:   1.0000  (100.0%)
Specificity:   0.1250  (12.5%)
AUROC:         0.8586
F1 (Abnormal): 0.7308

Confusion Matrix:
  TP=19  FP=14
  FN=0  TN=2

Pre-registered criteria:
  Sensitivity >= 80%: PASS
  Specificity >= 75%: FAIL

No amber-tier subjects.


## Cell 6 — Tier Assessment

Final tier evaluation for MIMIC PERform AF Layer 2 validation:
- **Tier 1:** Pipeline executed and gap quantification completed
- **Tier 2:** Sensitivity >= 80% AND specificity >= 75%
- **Tier 3:** AUROC >= 0.85

**Output:** `outputs/layer2/tier_assessment_mimic.csv`

In [14]:
# src calls: none
# Output: outputs/layer2/tier_assessment_mimic.csv

# Tier 1: pipeline executed, gap quantification complete
tier_1 = "PASS" if len(features_df) > 0 else "FAIL"

# Tier 2: sensitivity >= 80% AND specificity >= 75%
tier_2 = "PASS" if (sensitivity >= 0.80 and specificity >= 0.75) else "FAIL"

# Tier 3: AUROC >= 0.85
tier_3 = "PASS" if auroc >= 0.85 else "FAIL"

tier_results = pd.DataFrame([
    {'tier': 'Tier 1 — Pipeline execution', 'result': tier_1},
    {'tier': 'Tier 2 — Sens >= 80% & Spec >= 75%', 'result': tier_2},
    {'tier': 'Tier 3 — AUROC >= 0.85', 'result': tier_3}
])

tier_results.to_csv(
    os.path.join(LAYER2_OUT, 'tier_assessment_mimic.csv'),
    index=False
)

print("Tier Assessment — MIMIC PERform AF")
print("=" * 55)
print(f"Tier 1 — Pipeline execution:          {tier_1}")
print(f"Tier 2 — Sens >= 80% & Spec >= 75%:   {tier_2}")
print(f"Tier 3 — AUROC >= 0.85:               {tier_3}")

Tier Assessment — MIMIC PERform AF
Tier 1 — Pipeline execution:          PASS
Tier 2 — Sens >= 80% & Spec >= 75%:   FAIL
Tier 3 — AUROC >= 0.85:               PASS


---

### Reading the Primary Result

Three numbers from Cell 5:
- **Sensitivity 100%** — the model flags every AF case. Zero false negatives.
- **Specificity 12.5%** — the model also flags nearly every NSR case. High false positive rate.
- **AUROC 0.8586** — the model retains meaningful discriminative ability between AF and NSR.

These three numbers tell a coherent, mechanistically explained story.

Cell 3 (gap quantification) showed all 8 features shifted toward values the model associates with Abnormal — MIMIC subjects are critically ill ICU patients with elevated HRV variability. The fixed threshold of 0.34 was calibrated on a Physionet population ranging from healthy to abnormal. When applied to a population whose baseline already looks abnormal to the model, almost everything crosses threshold.

**The specificity failure is a threshold calibration problem, not a discriminative failure.** AUROC 0.86 means the model can rank AF above NSR — it just cannot place the decision boundary at the right absolute probability.

**Cells 7–10 investigate what happens when the threshold is adapted to this domain.**

## Cell 7 — Cross-Model Comparison on MIMIC PERform AF

Runs all four Layer 1 models against the MIMIC PERform AF feature matrix at their fixed Layer 1 thresholds.
No retraining, no threshold adjustment — strict forward pass per model.

Each model uses its own validation-optimised threshold from `evaluation_report.json`.

**Input:** `outputs/layer2/mimic_perform_af_features.csv`, `outputs/models/scaler.joblib`, all four model joblib files
**Output:** `outputs/layer2/cross_model_comparison_mimic.csv`

In [15]:
# src calls: none
# Input: outputs/layer2/mimic_perform_af_features.csv, outputs/models/*.joblib
# Output: outputs/layer2/cross_model_comparison_mimic.csv

import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

# ── 1. Load feature matrix ────────────────────────────────────
mimic_df = pd.read_csv(os.path.join(LAYER2_OUT, 'mimic_perform_af_features.csv'))

feature_cols = ['rmssd', 'sdnn', 'mean_rr', 'pnn50',
                'hr_mean', 'hr_std', 'rr_skewness', 'rr_kurtosis']

X = mimic_df[feature_cols].values
y = mimic_df['label'].values

# ── 2. Scale — transform() only ──────────────────────────────
scaler_cm = joblib.load(os.path.join('outputs', 'models', 'scaler.joblib'))
X_scaled = scaler_cm.transform(X)

# ── 3. Model registry ────────────────────────────────────────
model_registry = [
    {"name": "Logistic Regression", "file": "logistic_regression.joblib", "threshold": 0.41},
    {"name": "Random Forest",       "file": "random_forest.joblib",       "threshold": 0.37},
    {"name": "XGBoost",             "file": "xgboost.joblib",             "threshold": 0.34},
    {"name": "SVM",                 "file": "selected_model.joblib",      "threshold": 0.34},
]

# ── 4. Evaluate each model ───────────────────────────────────
results = []

for entry in model_registry:
    mdl = joblib.load(os.path.join('outputs', 'models', entry['file']))
    proba = mdl.predict_proba(X_scaled)[:, 1]
    preds = (proba >= entry['threshold']).astype(int)

    cm = confusion_matrix(y, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    auroc = roc_auc_score(y, proba)
    f1 = f1_score(y, preds)

    results.append({
        'model':       entry['name'],
        'threshold':   entry['threshold'],
        'sensitivity': round(sens, 4),
        'specificity': round(spec, 4),
        'auroc':       round(auroc, 4),
        'f1':          round(f1, 4),
        'TP': int(tp), 'FP': int(fp),
        'FN': int(fn), 'TN': int(tn)
    })

# ── 5. Build and display results ─────────────────────────────
results_df = pd.DataFrame(results)

print("Cross-Model Comparison — MIMIC PERform AF (Fixed Thresholds)")
print("=" * 80)
print(results_df.to_string(index=False))

# ── 6. Save ──────────────────────────────────────────────────
results_df.to_csv(
    os.path.join(LAYER2_OUT, 'cross_model_comparison_mimic.csv'),
    index=False
)
print(f"\nSaved to outputs/layer2/cross_model_comparison_mimic.csv")

Cross-Model Comparison — MIMIC PERform AF (Fixed Thresholds)
              model  threshold  sensitivity  specificity  auroc     f1  TP  FP  FN  TN
Logistic Regression       0.41          1.0       0.1875 0.7368 0.7451  19  13   0   3
      Random Forest       0.37          1.0       0.0625 0.8503 0.7170  19  15   0   1
            XGBoost       0.34          1.0       0.1250 0.8322 0.7308  19  14   0   2
                SVM       0.34          1.0       0.1250 0.8586 0.7308  19  14   0   2

Saved to outputs/layer2/cross_model_comparison_mimic.csv


## Cell 8 — Threshold Recalibration Analysis (SVM, MIMIC PERform AF)

Computes the ROC curve for the SVM on MIMIC data and finds the Youden's J optimal threshold.
This answers: how much specificity recovers if the threshold is recalibrated to the target domain.

No retraining — only the decision threshold changes. This is a post-hoc analytical step,
not a modification to the deployed model.

**Input:** `outputs/layer2/mimic_perform_af_features.csv`, `outputs/models/scaler.joblib`, `outputs/models/selected_model.joblib`
**Output:** `outputs/layer2/threshold_recalibration_mimic.csv`

In [16]:
# src calls: none
# Input: outputs/layer2/mimic_perform_af_features.csv, outputs/models/scaler.joblib, selected_model.joblib
# Output: outputs/layer2/threshold_recalibration_mimic.csv

import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

from sklearn.metrics import roc_curve

# ── 1. Load feature matrix ────────────────────────────────────
mimic_df = pd.read_csv(os.path.join(LAYER2_OUT, 'mimic_perform_af_features.csv'))

feature_cols = ['rmssd', 'sdnn', 'mean_rr', 'pnn50',
                'hr_mean', 'hr_std', 'rr_skewness', 'rr_kurtosis']

X = mimic_df[feature_cols].values
y = mimic_df['label'].values

# ── 2. Scale — transform() only ──────────────────────────────
scaler_rc = joblib.load(os.path.join('outputs', 'models', 'scaler.joblib'))
X_scaled = scaler_rc.transform(X)

# ── 3. Load SVM, get probabilities ───────────────────────────
svm_model = joblib.load(os.path.join('outputs', 'models', 'selected_model.joblib'))
proba = svm_model.predict_proba(X_scaled)[:, 1]

# ── 4. Compute ROC curve ─────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y, proba)

# ── 5. Youden's J statistic ──────────────────────────────────
sensitivity_arr = tpr
specificity_arr = 1 - fpr
j_scores = sensitivity_arr + specificity_arr - 1

optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]
optimal_sens = sensitivity_arr[optimal_idx]
optimal_spec = specificity_arr[optimal_idx]

# ── 6. Confusion matrix at optimal threshold ──────────────────
optimal_preds = (proba >= optimal_threshold).astype(int)
cm_opt = confusion_matrix(y, optimal_preds, labels=[0, 1])
tn_opt, fp_opt, fn_opt, tp_opt = cm_opt.ravel()
f1_opt = f1_score(y, optimal_preds)

# ── Fixed threshold results for comparison ────────────────────
FIXED_THRESHOLD = 0.34
fixed_preds = (proba >= FIXED_THRESHOLD).astype(int)
cm_fix = confusion_matrix(y, fixed_preds, labels=[0, 1])
tn_fix, fp_fix, fn_fix, tp_fix = cm_fix.ravel()
fixed_sens = tp_fix / (tp_fix + fn_fix) if (tp_fix + fn_fix) > 0 else 0.0
fixed_spec = tn_fix / (tn_fix + fp_fix) if (tn_fix + fp_fix) > 0 else 0.0
f1_fix = f1_score(y, fixed_preds)

# ── 7. Print comparison table ─────────────────────────────────
print("Threshold Comparison — SVM on MIMIC PERform AF")
print("=" * 55)
print(f"{'Metric':<20} {'Fixed (0.34)':>15} {'Recalibrated':>15}")
print("-" * 55)
print(f"{'Threshold':<20} {FIXED_THRESHOLD:>15.2f} {optimal_threshold:>15.4f}")
print(f"{'Sensitivity':<20} {fixed_sens:>14.1%} {optimal_sens:>14.1%}")
print(f"{'Specificity':<20} {fixed_spec:>14.1%} {optimal_spec:>14.1%}")
print(f"{'F1':<20} {f1_fix:>15.4f} {f1_opt:>15.4f}")
print(f"{'TP/FP/FN/TN':<20} {tp_fix:>3}/{fp_fix:>2}/{fn_fix:>2}/{tn_fix:>2}"
      f"       {tp_opt:>3}/{fp_opt:>2}/{fn_opt:>2}/{tn_opt:>2}")
print("=" * 55)
print(f"\nYouden's J at optimal: {j_scores[optimal_idx]:.4f}")

# ── 8. Save ──────────────────────────────────────────────────
recal_df = pd.DataFrame([{
    'model':              'SVM',
    'fixed_threshold':    FIXED_THRESHOLD,
    'fixed_sens':         round(fixed_sens, 4),
    'fixed_spec':         round(fixed_spec, 4),
    'fixed_f1':           round(f1_fix, 4),
    'optimal_threshold':  round(float(optimal_threshold), 4),
    'optimal_sens':       round(float(optimal_sens), 4),
    'optimal_spec':       round(float(optimal_spec), 4),
    'optimal_f1':         round(f1_opt, 4),
    'fixed_TP': int(tp_fix),  'fixed_FP': int(fp_fix),
    'fixed_FN': int(fn_fix),  'fixed_TN': int(tn_fix),
    'optimal_TP': int(tp_opt), 'optimal_FP': int(fp_opt),
    'optimal_FN': int(fn_opt), 'optimal_TN': int(tn_opt)
}])

recal_df.to_csv(
    os.path.join(LAYER2_OUT, 'threshold_recalibration_mimic.csv'),
    index=False
)
print(f"\nSaved to outputs/layer2/threshold_recalibration_mimic.csv")

Threshold Comparison — SVM on MIMIC PERform AF
Metric                  Fixed (0.34)    Recalibrated
-------------------------------------------------------
Threshold                       0.34          0.8424
Sensitivity                  100.0%          73.7%
Specificity                   12.5%          93.8%
F1                            0.7308          0.8235
TP/FP/FN/TN           19/14/ 0/ 2        14/ 1/ 5/15

Youden's J at optimal: 0.6743

Saved to outputs/layer2/threshold_recalibration_mimic.csv


## Cell 9 — Stress Testing (SVM, MIMIC PERform AF)

Three robustness checks on SVM inference results from MIMIC PERform AF:

1. **Bootstrap CI on AUROC** — 1,000 resamples with replacement. Reports mean and 95% CI.
2. **Bootstrap CI on recalibrated metrics** — same 1,000 resamples, each with its own Youden's J threshold. Reports 95% CI for sensitivity, specificity, F1.
3. **LOOCV on recalibrated threshold** — leave-one-out cross-validation where the optimal threshold is recomputed on 34 subjects and applied to the held-out subject. Tests whether recalibrated specificity is robust to single-subject removal.

No retraining, no fitting on full dataset. SVM probabilities are fixed — only the threshold and resampling vary.

**Input:** `outputs/layer2/mimic_perform_af_features.csv`, `outputs/models/scaler.joblib`, `outputs/models/selected_model.joblib`
**Output:** `outputs/layer2/stress_test_results_mimic.csv`

In [17]:
# src calls: none
# Input: outputs/layer2/mimic_perform_af_features.csv, outputs/models/scaler.joblib, selected_model.joblib
# Output: outputs/layer2/stress_test_results_mimic.csv

import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

from sklearn.metrics import roc_curve, roc_auc_score, f1_score, confusion_matrix

RANDOM_STATE = 42
N_BOOTSTRAP = 1000

# ── Setup ─────────────────────────────────────────────────────
mimic_df = pd.read_csv(os.path.join(LAYER2_OUT, 'mimic_perform_af_features.csv'))

feature_cols = ['rmssd', 'sdnn', 'mean_rr', 'pnn50',
                'hr_mean', 'hr_std', 'rr_skewness', 'rr_kurtosis']

X = mimic_df[feature_cols].values
y = mimic_df['label'].values

scaler_st = joblib.load(os.path.join('outputs', 'models', 'scaler.joblib'))
X_scaled = scaler_st.transform(X)

svm_model = joblib.load(os.path.join('outputs', 'models', 'selected_model.joblib'))
scores = svm_model.predict_proba(X_scaled)[:, 1]

n = len(y)
rng = np.random.RandomState(RANDOM_STATE)

# ══════════════════════════════════════════════════════════════
# Test 1 — Bootstrap CI on AUROC
# ══════════════════════════════════════════════════════════════
auroc_boot = []
for _ in range(N_BOOTSTRAP):
    idx = rng.randint(0, n, size=n)
    if len(np.unique(y[idx])) < 2:
        continue
    auroc_boot.append(roc_auc_score(y[idx], scores[idx]))

auroc_boot = np.array(auroc_boot)
auroc_mean = auroc_boot.mean()
auroc_ci_lo = np.percentile(auroc_boot, 2.5)
auroc_ci_hi = np.percentile(auroc_boot, 97.5)

print("--- Test 1: Bootstrap AUROC (n=1000) ---")
print(f"Mean AUROC: {auroc_mean:.4f}")
print(f"95% CI: [{auroc_ci_lo:.4f}, {auroc_ci_hi:.4f}]")

# ══════════════════════════════════════════════════════════════
# Test 2 — Bootstrap CI on Recalibrated Metrics
# ══════════════════════════════════════════════════════════════
rng2 = np.random.RandomState(RANDOM_STATE)
boot_sens, boot_spec, boot_f1 = [], [], []

for _ in range(N_BOOTSTRAP):
    idx = rng2.randint(0, n, size=n)
    y_b = y[idx]
    s_b = scores[idx]
    if len(np.unique(y_b)) < 2:
        continue

    fpr_b, tpr_b, thresh_b = roc_curve(y_b, s_b)
    j_b = tpr_b + (1 - fpr_b) - 1
    opt_idx = np.argmax(j_b)
    opt_thresh = thresh_b[opt_idx]

    preds_b = (s_b >= opt_thresh).astype(int)
    cm_b = confusion_matrix(y_b, preds_b, labels=[0, 1])
    tn_b, fp_b, fn_b, tp_b = cm_b.ravel()

    sens_b = tp_b / (tp_b + fn_b) if (tp_b + fn_b) > 0 else 0.0
    spec_b = tn_b / (tn_b + fp_b) if (tn_b + fp_b) > 0 else 0.0
    f1_b = f1_score(y_b, preds_b, zero_division=0)

    boot_sens.append(sens_b)
    boot_spec.append(spec_b)
    boot_f1.append(f1_b)

boot_sens = np.array(boot_sens)
boot_spec = np.array(boot_spec)
boot_f1 = np.array(boot_f1)

print(f"\n--- Test 2: Bootstrap Recalibrated Metrics (n=1000) ---")
print(f"{'':14} {'Mean':>8}    {'95% CI':>20}")
print(f"{'Sensitivity':<14} {boot_sens.mean():>7.1%}    "
      f"[{np.percentile(boot_sens, 2.5):.1%}, {np.percentile(boot_sens, 97.5):.1%}]")
print(f"{'Specificity':<14} {boot_spec.mean():>7.1%}    "
      f"[{np.percentile(boot_spec, 2.5):.1%}, {np.percentile(boot_spec, 97.5):.1%}]")
print(f"{'F1':<14} {boot_f1.mean():>8.4f}    "
      f"[{np.percentile(boot_f1, 2.5):.4f}, {np.percentile(boot_f1, 97.5):.4f}]")

# ══════════════════════════════════════════════════════════════
# Test 3 — LOOCV on Recalibrated Threshold
# ══════════════════════════════════════════════════════════════
loocv_preds = np.zeros(n, dtype=int)

for i in range(n):
    mask = np.ones(n, dtype=bool)
    mask[i] = False

    y_train_loo = y[mask]
    s_train_loo = scores[mask]

    if len(np.unique(y_train_loo)) < 2:
        loocv_preds[i] = 1
        continue

    fpr_loo, tpr_loo, thresh_loo = roc_curve(y_train_loo, s_train_loo)
    j_loo = tpr_loo + (1 - fpr_loo) - 1
    opt_idx_loo = np.argmax(j_loo)
    opt_thresh_loo = thresh_loo[opt_idx_loo]

    loocv_preds[i] = int(scores[i] >= opt_thresh_loo)

cm_loo = confusion_matrix(y, loocv_preds, labels=[0, 1])
tn_loo, fp_loo, fn_loo, tp_loo = cm_loo.ravel()

sens_loo = tp_loo / (tp_loo + fn_loo) if (tp_loo + fn_loo) > 0 else 0.0
spec_loo = tn_loo / (tn_loo + fp_loo) if (tn_loo + fp_loo) > 0 else 0.0
f1_loo = f1_score(y, loocv_preds, zero_division=0)

print(f"\n--- Test 3: LOOCV Recalibrated Threshold ---")
print(f"Sensitivity:  {sens_loo:.1%}  (TP={tp_loo}, FN={fn_loo})")
print(f"Specificity:  {spec_loo:.1%}  (TN={tn_loo}, FP={fp_loo})")
print(f"F1:           {f1_loo:.4f}")

# ══════════════════════════════════════════════════════════════
# Save
# ══════════════════════════════════════════════════════════════
stress_rows = [
    {'test': 'Bootstrap AUROC',       'metric': 'auroc',       'value': round(auroc_mean, 4),
     'ci_lower': round(auroc_ci_lo, 4), 'ci_upper': round(auroc_ci_hi, 4)},
    {'test': 'Bootstrap Recalibrated', 'metric': 'sensitivity', 'value': round(float(boot_sens.mean()), 4),
     'ci_lower': round(float(np.percentile(boot_sens, 2.5)), 4),
     'ci_upper': round(float(np.percentile(boot_sens, 97.5)), 4)},
    {'test': 'Bootstrap Recalibrated', 'metric': 'specificity', 'value': round(float(boot_spec.mean()), 4),
     'ci_lower': round(float(np.percentile(boot_spec, 2.5)), 4),
     'ci_upper': round(float(np.percentile(boot_spec, 97.5)), 4)},
    {'test': 'Bootstrap Recalibrated', 'metric': 'f1',          'value': round(float(boot_f1.mean()), 4),
     'ci_lower': round(float(np.percentile(boot_f1, 2.5)), 4),
     'ci_upper': round(float(np.percentile(boot_f1, 97.5)), 4)},
    {'test': 'LOOCV Recalibrated',    'metric': 'sensitivity', 'value': round(sens_loo, 4),
     'ci_lower': None, 'ci_upper': None},
    {'test': 'LOOCV Recalibrated',    'metric': 'specificity', 'value': round(spec_loo, 4),
     'ci_lower': None, 'ci_upper': None},
    {'test': 'LOOCV Recalibrated',    'metric': 'f1',          'value': round(f1_loo, 4),
     'ci_lower': None, 'ci_upper': None},
]

stress_df = pd.DataFrame(stress_rows)
stress_df.to_csv(
    os.path.join(LAYER2_OUT, 'stress_test_results_mimic.csv'),
    index=False
)
print(f"\nSaved to outputs/layer2/stress_test_results_mimic.csv")

--- Test 1: Bootstrap AUROC (n=1000) ---
Mean AUROC: 0.8631
95% CI: [0.7246, 0.9720]

--- Test 2: Bootstrap Recalibrated Metrics (n=1000) ---
                   Mean                  95% CI
Sensitivity      80.6%    [57.1%, 100.0%]
Specificity      91.3%    [72.7%, 100.0%]
F1               0.8534    [0.7059, 0.9631]

--- Test 3: LOOCV Recalibrated Threshold ---
Sensitivity:  68.4%  (TP=13, FN=6)
Specificity:  81.2%  (TN=13, FP=3)
F1:           0.7429

Saved to outputs/layer2/stress_test_results_mimic.csv


## Cell 10 — Sensitivity-Targeted LOOCV (SVM, MIMIC PERform AF)

Alternative LOOCV where each fold selects the most conservative threshold that still achieves
sensitivity >= 80% on the 34 training subjects, then applies it to the held-out subject.

This mirrors the Layer 1 threshold selection logic (maximise specificity subject to sensitivity floor)
and tests whether a screening-appropriate threshold generalises across the MIMIC population.

If no threshold achieves 80% sensitivity on a fold, Youden's J is used as fallback.

**Input:** `outputs/layer2/mimic_perform_af_features.csv`, `outputs/models/scaler.joblib`, `outputs/models/selected_model.joblib`
**Output:** `outputs/layer2/sensitivity_targeted_loocv_mimic.csv`

In [18]:
# src calls: none
# Input: outputs/layer2/mimic_perform_af_features.csv, outputs/models/scaler.joblib, selected_model.joblib
# Output: outputs/layer2/sensitivity_targeted_loocv_mimic.csv

import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

from sklearn.metrics import roc_curve, f1_score, confusion_matrix

# ── Setup ─────────────────────────────────────────────────────
mimic_df = pd.read_csv(os.path.join(LAYER2_OUT, 'mimic_perform_af_features.csv'))

feature_cols = ['rmssd', 'sdnn', 'mean_rr', 'pnn50',
                'hr_mean', 'hr_std', 'rr_skewness', 'rr_kurtosis']

X = mimic_df[feature_cols].values
y = mimic_df['label'].values

scaler_st = joblib.load(os.path.join('outputs', 'models', 'scaler.joblib'))
X_scaled = scaler_st.transform(X)

svm_model = joblib.load(os.path.join('outputs', 'models', 'selected_model.joblib'))
scores = svm_model.predict_proba(X_scaled)[:, 1]

n = len(y)

# ── LOOCV ─────────────────────────────────────────────────────
fold_records = []

for i in range(n):
    mask = np.ones(n, dtype=bool)
    mask[i] = False

    y_train = y[mask]
    s_train = scores[mask]
    fallback = False

    if len(np.unique(y_train)) < 2:
        # Edge case — only one class in training fold
        thresh_used = 0.5
        fallback = True
    else:
        fpr_f, tpr_f, thresh_f = roc_curve(y_train, s_train)

        # Sensitivity-targeted: highest threshold where tpr >= 0.80
        sens_mask = tpr_f >= 0.80
        if sens_mask.any():
            # Among qualifying points, take the highest threshold (most conservative)
            qualifying_idx = np.where(sens_mask)[0]
            # roc_curve returns thresholds in decreasing order for the first elements
            # tpr increases as threshold decreases, so the first qualifying index
            # corresponds to the highest threshold meeting the sensitivity floor
            best_idx = qualifying_idx[0]
            thresh_used = thresh_f[best_idx]
        else:
            # Fallback to Youden's J
            j_f = tpr_f + (1 - fpr_f) - 1
            best_idx = np.argmax(j_f)
            thresh_used = thresh_f[best_idx]
            fallback = True

    prediction = int(scores[i] >= thresh_used)

    fold_records.append({
        'subject_idx': i,
        'true_label':  int(y[i]),
        'prediction':  prediction,
        'threshold_used': round(float(thresh_used), 6),
        'fallback':    fallback
    })

# ── Aggregate results ─────────────────────────────────────────
fold_df = pd.DataFrame(fold_records)

y_true_loo = fold_df['true_label'].values
y_pred_loo = fold_df['prediction'].values

cm_loo = confusion_matrix(y_true_loo, y_pred_loo, labels=[0, 1])
tn_loo, fp_loo, fn_loo, tp_loo = cm_loo.ravel()

sens_loo = tp_loo / (tp_loo + fn_loo) if (tp_loo + fn_loo) > 0 else 0.0
spec_loo = tn_loo / (tn_loo + fp_loo) if (tn_loo + fp_loo) > 0 else 0.0
f1_loo = f1_score(y_true_loo, y_pred_loo, zero_division=0)

fallback_count = fold_df['fallback'].sum()
thresholds_used = fold_df['threshold_used'].values

# ── Print ─────────────────────────────────────────────────────
print("--- Cell 10: Sensitivity-Targeted LOOCV ---")
print("Target sensitivity: >= 80% on training fold")
print()
print("Per-fold threshold stats:")
print(f"  Mean threshold: {thresholds_used.mean():.4f}")
print(f"  Min threshold:  {thresholds_used.min():.4f}")
print(f"  Max threshold:  {thresholds_used.max():.4f}")
print(f"  Fallback folds: {fallback_count} of {n}")
print()
print("Aggregate results:")
print(f"  Sensitivity: {sens_loo:.1%}  (TP={tp_loo}, FN={fn_loo})")
print(f"  Specificity: {spec_loo:.1%}  (TN={tn_loo}, FP={fp_loo})")
print(f"  F1:          {f1_loo:.4f}")

# ── Save ──────────────────────────────────────────────────────
fold_df.to_csv(
    os.path.join(LAYER2_OUT, 'sensitivity_targeted_loocv_mimic.csv'),
    index=False
)
print(f"\nSaved to outputs/layer2/sensitivity_targeted_loocv_mimic.csv")

--- Cell 10: Sensitivity-Targeted LOOCV ---
Target sensitivity: >= 80% on training fold

Per-fold threshold stats:
  Mean threshold: 0.8368
  Min threshold:  0.8367
  Max threshold:  0.8414
  Fallback folds: 0 of 35

Aggregate results:
  Sensitivity: 78.9%  (TP=15, FN=4)
  Specificity: 81.2%  (TN=13, FP=3)
  F1:          0.8108

Saved to outputs/layer2/sensitivity_targeted_loocv_mimic.csv


---

## Answer to the Research Question

**Can machine learning models trained on clinical ECG data generalise to consumer wearable signals to detect cardiac rhythm anomalies?**

**Yes — with threshold adaptation.**

The discriminative signal transfers across modalities (AUROC 0.8586). The fixed threshold calibrated on ECG does not transfer to PPG (specificity 12.5%). When the threshold is recalibrated via out-of-sample LOOCV:

| Method | Sensitivity | Specificity |
|---|---|---|
| Fixed threshold (0.34) | 100% | 12.5% |
| Youden's J LOOCV | 68.4% | 81.2% |
| Sensitivity-targeted LOOCV | 78.9% | 81.2% |
| Pre-registered criterion | ≥ 80% | ≥ 75% |

LOOCV specificity (81.2%) clears the pre-registered criterion. Sensitivity (78.9%) is 1.1 percentage points below — consistent with the variance from N=19 AF cases in leave-one-out evaluation.

---

**Are consumer wearables feasible as proxy cardiac screening tools for the general population?**

**Feasible, with domain adaptation as the primary remaining obstacle.**

The model retains genuine discriminative ability across sensing modalities. Direct application of a clinically-calibrated threshold produces too many false positives. A domain-adapted threshold recovers specificity without sacrificing meaningful sensitivity. The BeatCheck application (see \) implements this approach using the LOOCV mean threshold (0.8368) as the out-of-sample estimate.

The modality gap — confirmed by large KS distances across all 8 features — is real but bridgeable. The primary challenge for real-world deployment is not discriminative ability. It is building the labelled wearable datasets needed to calibrate thresholds for each new device type.

---

*For Singapore public health context, study limitations, and recommendations for future work, see the final narrative report.*